# Cleaning the Clinical Data and Performing EDA:
The purpose of this notebook is to clean the multi-center clinical data. As the steps of cleaning were too specific for this dataset, a notebook is dedicated to it instead of a generic script, to explain the decisions taken. The final dataset includes the cleaned selected features and target columns for each research question. The target columns are as below:

For treatment response: **surv_bestresponse_car**  
For ICANS toxicity: **ae_summ_icans_highestgrade_v2 and ae_summ_icans_v2**  
For CRS toxicity: **ae_summ_crs_v2, ae_summ_highestgrade_v2**  

EDA is also performed using the ydata_profiling package to get an overview of the patients and the radiomic features.



## Cleaning Steps:

### Specifying the clinical feature subset
Not all the features are relevant for the analysis. The selection of features are clinically driven and are recorded before the pre-LD phase.

### Duplicate rows handeling
Some of the patients' data are duplicated in the clinical dataset, we need to remove them first.

### Intersect with the Radiomics dataset
We only want clinical data for which we have the radiomics features. **Note that patient umcg 95 is present in radiomics but not in clinical.**  

### Fix the clinical dataset inconsistancy
There are inconsistencies in how a Nan value is defined. Also some of the numeric values are wrongly formatted which would make it impossible to cast a numeric value on them.

### One Hot Encoding
The indication_dis_diagnosis which specifies the type of lymphoma is not one hot encoded originally.
 
### Type casting
all the features should be numerical/categorical, except for ID.  

### Handling missing values 
1. total_num_priortherapylines_fl: fill with 0, as it means they did not receive the therapy as it's a sign for Not Applicable.    
2. indication_extranodal_nr: fill based on indication_extran_invol, as it's only Nan for when the involvment is either 0 or 1.    
3. cli_st_neutrophils: only one missing, use median imputation, it's safe.    
4. ae_summ_icans_highestgrade_v2: fill with 0, the missing values are for when patients have not shown ICANS toxicity.    
5. indication_sec_refr: filled with "2", as in the original clinical data, the Nans are a group, encoded with 2, this one instance is not coded.  
6. indication_whops: only 1 missing, filled with median as it's safe.  
7. for cli_st_ldh (2%), cli_st_crp (10%) and cli_st_ferritin (31%), median is still a viable imputation method as there's not a large precentage of data missing.  
8. ae_summ_highestgrade_v2: fill based on ae_summ_crs_v2, if the value is Nan, it means the patient has not experienced any CRS toxicity and the ae_summ_crs_v2 is 0.
9. surv_bestresponse_car: fill with 0. The missing values are patients who passed away before the treatment response could be assessed. They will be assigned their own group, encoded as 0.  
### Renaming the bridging therapy columns  
Renaming the bridging regiement columns for better readability

In [13]:
import yaml
import pandas as pd
from pathlib import Path

# Path to config relative to notebook
config_path = Path("..") / "config.yaml"  
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

In [14]:
radiomics = {}

for name in ['A', 'B', 'delta']:
    radiomics[f"radiomics_{name}"] = pd.read_csv(config[name]['combined'], index_col=0)

In [15]:
clinical = pd.read_csv(config['clinical']['combined'])

In [16]:
# Specifying the clinical feature subset
clinical_ftrs = [
    "record_id",	
    "scr_sex",
    "scr_age",
    "scr_height",
    "scr_weight",
    "scr_bmi",
    "indication_dis_diagnosis.factor",
    "total_num_priortherapylines_fl",
    "total_num_priortherapylines_aggressive",
    "indication_priorsct",
    "indication_whops",
    "indication_ldh_uln",
    "indication_age_60",
    "indication_bulkydisease",
    "indication_stage",
    "indication_extran_invol",
    "indication_extranodal_nr",
    "indication_pri_refr",
    "indication_sec_refr",
    "indication_dis_lymsubtype_cns",
    "tr_car_br",
    "tr_car_bridg_type.factor",
    "cli_st_hemoglobin",
    "cli_st_trombocytes",
    "cli_st_leukocytes",
    "cli_st_neutrophils",
    "cli_st_ldh",
    "cli_st_crp",
    "cli_st_ferritin",
    "ae_summ_icans_highestgrade_v2",
    "ae_summ_icans_v2",
    "ae_summ_crs_v2",
    "ae_summ_highestgrade_v2",
    "surv_bestresponse_car"

]

In [17]:
clinical_filtered = clinical.loc[1: , clinical_ftrs]

In [18]:
# Dropping duplicates in the 'record_id' column
clinical_filtered.drop_duplicates(subset='record_id', inplace=True)

In [19]:
# Intersect with the Radiomics dataset
clinical_filtered = clinical_filtered[clinical_filtered['record_id'].isin(radiomics['radiomics_delta'].index)]

In [20]:
clinical_filtered.reset_index(drop=True, inplace=True)

In [21]:
# Fix the clinical dataset inconsistancies
clinical_filtered.replace(['NE','ne', 'NA','Not evaluated'], pd.NA, inplace=True)
clinical_filtered.replace('6,4', '6.4', inplace=True)
clinical_filtered.replace('5,0 ', '5.0', inplace=True)
clinical_filtered.replace('<0.5', '0.25', inplace=True) # Assuming <0.5 means 0.25 for the sake of numerical analysis
clinical_filtered['tr_car_bridg_type.factor'] = clinical_filtered['tr_car_bridg_type.factor'].replace(pd.NA, 'None', inplace=False)

In [22]:
# One Hot Encoding the 'indication_dis_diagnosis' column
clinical_filtered = pd.get_dummies(clinical_filtered, columns=['indication_dis_diagnosis.factor'], drop_first=False)

In [23]:
clinical_filtered = pd.get_dummies(clinical_filtered, columns=['tr_car_bridg_type.factor'], drop_first=False)

In [24]:
num_cols = [
    "scr_age",
    "scr_height",
    "scr_weight",
    "scr_bmi",
    "cli_st_hemoglobin",
    "cli_st_trombocytes",
    "cli_st_leukocytes",
    "cli_st_neutrophils",
    "cli_st_ldh",
    "cli_st_crp",
    "cli_st_ferritin"]

In [25]:
# Convert only numeric-expected columns
cat_cols = [col for col in clinical_filtered.columns.tolist() if col not in num_cols]
cat_cols.remove('record_id')  # Exclude 'record_id' from numeric conversion
all_cols = num_cols + cat_cols
clinical_filtered[all_cols] = clinical_filtered[all_cols].apply(
    pd.to_numeric, errors="raise"
).astype("float64")


In [26]:
# Handling missing values
clinical_filtered['total_num_priortherapylines_fl'] = clinical_filtered['total_num_priortherapylines_fl'].fillna(0)
clinical_filtered['indication_extranodal_nr'] = clinical_filtered['indication_extranodal_nr'].fillna(clinical_filtered['indication_extran_invol'])
clinical_filtered['ae_summ_icans_highestgrade_v2'] = clinical_filtered['ae_summ_icans_highestgrade_v2'].fillna(0)
clinical_filtered['indication_sec_refr'] = clinical_filtered['indication_sec_refr'].fillna(2)
clinical_filtered[['indication_whops', 'cli_st_ldh','cli_st_crp','cli_st_ferritin', 'cli_st_neutrophils']] = \
    clinical_filtered[['indication_whops', 'cli_st_ldh','cli_st_crp','cli_st_ferritin', 'cli_st_neutrophils']].fillna(clinical_filtered[['indication_whops', 'cli_st_ldh','cli_st_crp','cli_st_ferritin', 'cli_st_neutrophils']].median())
clinical_filtered['ae_summ_highestgrade_v2'] = clinical_filtered['ae_summ_highestgrade_v2'].fillna(clinical_filtered['ae_summ_crs_v2'])
clinical_filtered['surv_bestresponse_car'] = clinical_filtered['surv_bestresponse_car'].fillna(0)

In [27]:
# This is to convert the categorical columns to 'category' dtype for ydata_profiling to recognize them as categorical features in the profiling report
# also to save memory and optimize performance when working with categorical data
clinical_filtered[cat_cols] = clinical_filtered[cat_cols].astype('category')

In [28]:
clinical_filtered.isna().sum().sum()

np.int64(0)

In [29]:
clinical_filtered.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 41 columns):
 #   Column                                                             Non-Null Count  Dtype   
---  ------                                                             --------------  -----   
 0   record_id                                                          85 non-null     object  
 1   scr_sex                                                            85 non-null     category
 2   scr_age                                                            85 non-null     float64 
 3   scr_height                                                         85 non-null     float64 
 4   scr_weight                                                         85 non-null     float64 
 5   scr_bmi                                                            85 non-null     float64 
 6   total_num_priortherapylines_fl                                     85 non-null     category
 7   total_num_priorther

In [30]:
# save the cleaned clinical dataset, using pickle to preserve data types
clinical_filtered.to_pickle(config['clinical']['cleaned'])

## EDA: Clinical Data Profiling
We'll use the ydata_profiling package to get an overview of the clinical data. The exploratory analysis will be saved as an HTML file.

**Note**: The correlation method used by ydata profiler, is automatically selected based on each feature's data type [1].

**Note**: The correlation threshold for warning of high correlation is automatically set to 0.9. However, the document explains how to change it if needed [1].



----------------------------------

[1] https://docs.profiling.ydata.ai/4.5/advanced_settings/available_settings/#correlations

In [31]:
from ydata_profiling import ProfileReport

profile = ProfileReport(clinical_filtered, title="Clinical Data Profile")
# profile.to_notebook_iframe() # Uncomment this line to display the profile report in a Jupyter notebook
profile.to_file("clinical_data.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 41/41 [00:00<00:00, 238.57it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

# EDA: Radiomics Dataset

In [32]:
for name, df in radiomics.items():
    profile = ProfileReport(df, 
                            interactions={'continuous': False}, 
                            title=f"{name} Data Profile")
    profile.to_file(f"{name}_data.html")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 99/99 [00:00<00:00, 539968.92it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

...

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 99/99 [00:00<00:00, 237821.36it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]